In [1]:
import pandas as pd
from experiment_runner import LLMExperimentRunner
from experiment_prompts import get_gamblers_fallacy_experiments, get_loss_aversion_experiments, get_serial_position_effect_experiments, get_big_five_aspects_questionaire
from system_prompts import get_persona_system_prompt
from concurrent.futures import ThreadPoolExecutor

In [2]:
df = pd.read_csv("generated_data/Personas.csv")
df

,Gender,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,...,Neuroticism,Openness_Intellect,raw_persona,experiment_group,serial_position_effect_map_20,serial_position_effect_map_40,serial_position_effect_map_80,llm_persona,llm_persona_reasoning,persona_generation_date
0,male,2.31,3.65,3.33,3.39,3.76,4.75,3.14,2.97,3.92,...,3.06,3.94,Gender: male\nEnthusiasm: 2.31\nAssertiveness:...,2,"{""ladder"": 1, ""lantern"": 2, ""bridge"": 3, ""penc...","{""forest"": 1, ""jungle"": 2, ""garage"": 3, ""butte...","{""suitcase"": 1, ""pillow"": 2, ""quilt"": 3, ""cook...","In his day-to-day life, he moves with a quiet ...",We are asked to construct a detailed psycholog...,2026-08-04 17:31:23+02:00
1,male,2.86,2.94,3.28,3.29,2.91,3.74,3.28,3.17,3.70,...,3.22,3.42,Gender: male\nEnthusiasm: 2.86\nAssertiveness:...,1,"{""violin"": 1, ""jacket"": 2, ""drum"": 3, ""rocket""...","{""fountain"": 1, ""helmet"": 2, ""bottle"": 3, ""jac...","{""bottle"": 1, ""turtle"": 2, ""meadow"": 3, ""clock...","He values a life that runs smoothly, where thi...","We are given: Gender: male. Enthusiasm: 2.86, ...",2026-08-04 17:31:23+02:00


In [3]:
PERSONA_SYSTEM_PROMPT = get_persona_system_prompt()
runner = LLMExperimentRunner(
    llm="deepseek-v4-pro",
    max_workers=2,
    thinking='enabled',
    reasoning_effort="high",
    )
df_persona = runner.generate_personas(df, PERSONA_SYSTEM_PROMPT)
df_persona

Generating personas: 100%|██████████| 2/2 [01:02<00:00, 31.18s/it]


,Gender,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,...,Neuroticism,Openness_Intellect,raw_persona,experiment_group,serial_position_effect_map_20,serial_position_effect_map_40,serial_position_effect_map_80,llm_persona,llm_persona_reasoning,persona_generation_date
0,male,2.31,3.65,3.33,3.39,3.76,4.75,3.14,2.97,3.92,...,3.06,3.94,Gender: male\nEnthusiasm: 2.31\nAssertiveness:...,2,"{""ladder"": 1, ""lantern"": 2, ""bridge"": 3, ""penc...","{""forest"": 1, ""jungle"": 2, ""garage"": 3, ""butte...","{""suitcase"": 1, ""pillow"": 2, ""quilt"": 3, ""cook...","In his day-to-day life, he moves with a quiet ...",We are asked to construct a detailed psycholog...,2026-08-04 17:31:23+02:00
1,male,2.86,2.94,3.28,3.29,2.91,3.74,3.28,3.17,3.70,...,3.22,3.42,Gender: male\nEnthusiasm: 2.86\nAssertiveness:...,1,"{""violin"": 1, ""jacket"": 2, ""drum"": 3, ""rocket""...","{""fountain"": 1, ""helmet"": 2, ""bottle"": 3, ""jac...","{""bottle"": 1, ""turtle"": 2, ""meadow"": 3, ""clock...","He values a life that runs smoothly, where thi...","We are given: Gender: male. Enthusiasm: 2.86, ...",2026-08-04 17:31:23+02:00


In [5]:
word_count = len(df_persona.iloc[0]["llm_persona"].split())
print(word_count)

854


In [ ]:
def main_run_experiment(
    df,
    conditions,
    max_workers,
    use_checkpoint
):

    for experiment_name in conditions["experiments"]:
        experiment_results = []
        for persona in conditions["personas"]:
            for thinking in conditions["thinking_options"]:
                for llm in conditions["llms"]:

                    runner = LLMExperimentRunner(
                        llm=llm,
                        thinking=thinking,
                        reasoning_effort=conditions["reasoning_effort"] if thinking == "enabled" else None,
                        max_workers=max_workers,
                        use_checkpoint=use_checkpoint
                    )
                    print(f"Condition: experiment={experiment_name}, persona={persona}, llm={llm}, thinking={thinking}")
                    if experiment_name in ["loss_aversion", "gamblers_fallacy"]:

                        results = []
                        # Group 1 the instructions are in order AB and in Group 2 they are BA (the text of the instructions)
                        # Group 1 the instructions for the previousoutcomes are red and in Group 2 they are black
                        for group in [1,2]:
                            df_group = df[df["experiment_group"] == group].copy()

                            if experiment_name == "loss_aversion":
                                instructions = get_loss_aversion_experiments(
                                    persona=persona,
                                    group=group
                                )
                            else:
                                instructions = get_gamblers_fallacy_experiments(
                                    max_previous_outcomes=conditions["max_previous_outcomes"],
                                    persona=persona,
                                    group=group
                                )

                            result = runner.run_experiment(
                                df=df_group,
                                instructions=instructions,
                                persona=persona,
                                run_id=f"group_{group}"
                            )

                            results.append(result)

                        result = pd.concat(results).sort_index()

                    else:

                        if experiment_name == "serial_position_effect":
                            instructions = get_serial_position_effect_experiments(
                                persona
                            )

                        else:
                            instructions = get_big_five_aspects_questionaire(
                                persona
                            )

                        result = runner.run_experiment(
                            df=df,
                            instructions=instructions,
                            persona=persona,
                            run_id=""
                            )


                    result["llm"] = llm
                    result["persona"] = persona
                    result["thinking"] = thinking
                    experiment_results.append(result)


        final = pd.concat(experiment_results)

        final.to_csv(
            runner.data_dir /
            f"{experiment_name}_results.csv",
            index=False
        )

In [6]:
CONDITIONS = {
    "experiments": ["loss_aversion", "gamblers_fallacy", "serial_position_effect", "big_five_evaluation"],
    "personas": ["description", "raw_persona", "no_persona"],
    "thinking_options": ["enabled", "disabled"],
    "llms": ["deepseek-v4-pro", "deepseek-v4-flash"],
    "max_previous_outcomes": 10,
    "reasoning_effort": "high"

}
main_run_experiment(
    df=df,
    conditions=CONDITIONS,
    max_workers=2,
    use_checkpoint=False
)

Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-pro, thinking=enabled


Experiment:: 100%|██████████| 16/16 [02:49<00:00, 10.61s/it]


Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-flash, thinking=enabled


Experiment:: 100%|██████████| 16/16 [01:48<00:00,  6.80s/it]


Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-pro, thinking=disabled


Experiment:: 100%|██████████| 16/16 [00:18<00:00,  1.13s/it]


... (86 outputs left out)

Condition: experiment=big_five_evaluation, persona=no_persona, llm=deepseek-v4-pro, thinking=disabled


Experiment:: 100%|██████████| 40/40 [00:58<00:00,  1.45s/it]


Condition: experiment=big_five_evaluation, persona=no_persona, llm=deepseek-v4-flash, thinking=disabled


Experiment:: 100%|██████████| 40/40 [00:43<00:00,  1.10s/it]


test this below, runs both groups in parallel:

In [ ]:
def main_run_experiment(
    df,
    conditions,
    max_workers,
    use_checkpoint, 
    enable_logging
):

    for experiment_name in conditions["experiments"]:
        experiment_results = []
        for persona in conditions["personas"]:
            for thinking in conditions["thinking_options"]:
                for llm in conditions["llms"]:
                    workers = max(1, max_workers // 2) if experiment_name in ["loss_aversion", "gamblers_fallacy"] else max_workers
                    runner = LLMExperimentRunner(
                        llm=llm,
                        thinking=thinking,
                        reasoning_effort=conditions["reasoning_effort"] if thinking == "enabled" else None,
                        max_workers=workers,
                        use_checkpoint=use_checkpoint,
                        enable_logging=enable_logging
                    )
                    print(f"\nCondition: experiment={experiment_name}, persona={persona}, llm={llm}, thinking={thinking}")
                    
                    if experiment_name in ["loss_aversion", "gamblers_fallacy"]:
                        # Group 1 the instructions are in order AB and in Group 2 they are BA (the text of the instructions)
                        # Group 1 the instructions for the previousoutcomes are red and in Group 2 they are black

                        def run_group(group):
                            df_group = df[df["experiment_group"] == group].copy()

                            if experiment_name == "loss_aversion":
                                instructions = get_loss_aversion_experiments(
                                    persona=persona,
                                    group=group
                                )
                            else:
                                instructions = get_gamblers_fallacy_experiments(
                                    max_previous_outcomes=conditions["max_previous_outcomes"],
                                    persona=persona,
                                    group=group
                                )

                            return runner.run_experiment(
                                df=df_group,
                                instructions=instructions,
                                persona=persona,
                                run_id=f"group_{group}",
                            )

                        with ThreadPoolExecutor(max_workers=2) as executor:
                            results = list(executor.map(run_group, [1, 2]))

                        result = pd.concat(results).sort_index()

                    else:

                        if experiment_name == "serial_position_effect":
                            instructions = get_serial_position_effect_experiments(
                                persona
                            )

                        else:
                            instructions = get_big_five_aspects_questionaire(
                                persona
                            )

                        result = runner.run_experiment(
                            df=df,
                            instructions=instructions,
                            persona=persona,
                            run_id=""
                            )


                    result["llm"] = llm
                    result["persona"] = persona
                    result["thinking"] = thinking
                    experiment_results.append(result)


        final = pd.concat(experiment_results)

        final.to_csv(
            runner.data_dir /
            f"{experiment_name}_results.csv",
            index=False
        )

In [4]:
CONDITIONS = {
    "experiments": ["loss_aversion", "gamblers_fallacy", "serial_position_effect", "big_five_evaluation"],
    "personas": ["description", "raw_persona", "no_persona"],
    "thinking_options": ["enabled", "disabled"],
    "llms": ["deepseek-v4-pro", "deepseek-v4-flash"],
    "max_previous_outcomes": 10,
    "reasoning_effort": "high"

}
main_run_experiment(
    df=df,
    conditions=CONDITIONS,
    max_workers=2,
    use_checkpoint=False,
    enable_logging=True
)

Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-pro, thinking=enabled


Experimentgroup_1:   0%|          | 0/16 [00:00<?, ?it/s]

Experimentgroup_1: 100%|██████████| 16/16 [07:00<00:00, 26.26s/it]


Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-flash, thinking=enabled


Experimentgroup_1: 100%|██████████| 16/16 [03:36<00:00, 13.51s/it]


Condition: experiment=loss_aversion, persona=description, llm=deepseek-v4-pro, thinking=disabled


... (87 outputs left out)

Condition: experiment=big_five_evaluation, persona=no_persona, llm=deepseek-v4-pro, thinking=disabled


Experiment: 100%|██████████| 80/80 [01:06<00:00,  1.20it/s]


Condition: experiment=big_five_evaluation, persona=no_persona, llm=deepseek-v4-flash, thinking=disabled


Experiment: 100%|██████████| 80/80 [00:54<00:00,  1.48it/s]


In [ ]:
CONDITIONS = {
    "experiments": ["loss_aversion", "gamblers_fallacy", "serial_position_effect", "big_five_evaluation"],
    "personas": ["description", "raw_persona", "no_persona"],
    "thinking_options": ["enabled", "disabled"],
    "llms": ["deepseek-v4-pro", "deepseek-v4-flash"],
    "max_previous_outcomes": 10,
    "reasoning_effort": "high"

}
main_run_experiment(
    df=df,
    conditions=CONDITIONS,
    max_workers=2,
    use_checkpoint=True,
    enable_logging=False
)

In [ ]:
# #test
# def main_run_experiment(
#     df,
#     conditions,
#     max_workers,
#     use_checkpoint
# ):
#     llm = "deepseek-v4-pro"
#     thinking = "enabled"
#     for experiment_name in conditions["experiments"]:
#         experiment_results = []
#         for persona in conditions["personas"]:


#                     runner = LLMExperimentRunner(
#                         llm=llm,
#                         thinking=thinking,
#                         reasoning_effort=conditions["reasoning_effort"] if thinking == "enabled" else None,
#                         max_workers=max_workers,
#                         use_checkpoint=use_checkpoint
#                     )
#                     print(f"Condtion: {experiment_name}_{persona}_{llm}_thinking_{thinking}")
#                     if experiment_name in ["loss_aversion", "gamblers_fallacy"]:

#                         results = []
#                         # Group 1 the instructions are in order AB and in Group 2 they are BA (the text of the instructions)
#                         # Group 1 the instructions for the previousoutcomes are red and in Group 2 they are black
#                         for group in [1,2]:
#                             df_group = df[df["experiment_group"] == group].copy()

#                             if experiment_name == "loss_aversion":
#                                 instructions = get_loss_aversion_experiments(
#                                     persona=persona,
#                                     group=group
#                                 )
#                             else:
#                                 instructions = get_gamblers_fallacy_experiments(
#                                     max_previous_outcomes=conditions["max_previous_outcomes"],
#                                     persona=persona,
#                                     group=group
#                                 )

#                             result = runner.run_experiment(
#                                 df=df_group,
#                                 instructions=instructions,
#                                 persona=persona,
#                                 run_id=f"group_{group}"
#                             )

#                             results.append(result)

#                         result = pd.concat(results).sort_index()

#                     else:

#                         if experiment_name == "serial_position_effect":
#                             instructions = get_serial_position_effect_experiments(
#                                 persona
#                             )

#                         else:
#                             instructions = get_big_five_aspects_questionaire(
#                                 persona
#                             )

#                         result = runner.run_experiment(
#                             df=df,
#                             instructions=instructions,
#                             persona=persona,
#                             run_id=""
#                             )


#                     result["experiment"] = experiment_name
#                     result["llm"] = llm
#                     result["persona"] = persona
#                     result["thinking"] = thinking
#                     experiment_results.append(result)


#         final = pd.concat(experiment_results)

#         final.to_csv(
#             runner.data_dir /
#             f"{experiment_name}_results.csv",
#             index=False
#         )